# How to Describe What a Bundle and Its Timelines Stand For

A TimeToShare manifest holds two planes. Its temporal structure states how
coordinates relate: timelines, their nesting, maps, regions and flow
control. Its {{< glossary statement >}}s state what the elements stand
for and how those things relate to others, and never nest, order or align
anything.

Three kinds of element are subjects of statements: the bundle, which is
the manifest itself and names the work it is about; every timeline, at any
depth, for the manifestation it is the axis of (an edition, an encoding, a
recording file, a video); and every region, for the stretch of that
manifestation it marks. A statement is a pair of a property and a value,
both identifiers in published vocabularies where they name things; the
property `type` reads "is a". Works and expressions are never elements:
they appear as values, the work under the bundle's `schema:about`.

A manifest carrying statements is YAML-LD: its inline `@context` declares
the prefixes its compact IRIs use, so it reads as linked data without
fetching anything. The {{< glossary catalogue >}} shipped with the
TimeToShare package knows the vocabularies, the identifier systems and the
verbs that state one relation in each of them.

This guide describes the LayerIt example: an MEI encoding of the first six
bars of Debussy's *Clair de lune*, a concert video of Maria João Pires and
a WAV file cut from its sound. It states what each timeline is, who
performs, who said so, decides on proposals, describes a region, writes
the manifest, and imports the description of an MEI header and of a
MeLODy repository. The format is specified in TimeToShare's
[description of a manifest](https://github.com/TimeToAlign/TimeToShare/blob/main/docs/description.md).

## Setup

Install the library with `pip install timetoalign`; the MeLODy import also
needs rdflib, `pip install "timetoalign[rdf]"`. No data is downloaded: the
MEI header and the MeLODy repository are written below into a temporary
directory. The identifiers are those of public catalogues; the editor who
states the pairs and the days are invented.

`show` prints the pairs of a subject, or of a value node inside it, and
the {{< glossary "provenance entry" >}} of each pair that has one; a value
node's own entries are shown when `show` is given that node.
`show_manifest` prints a manifest as canonical YAML, leaving the format's
own terms out of `@context` so that only the prefixes remain.

In [1]:
import json
import tempfile
from pathlib import Path

from timetoshare import FORMAT_CONTEXT, dumps, to_file_id

import timetoalign as tta
from timetoalign.loader import MeiHeadLoader, MelodyLoader

EDITOR = "https://example.org/people/editor"
DAY = "2026-11-18"
FOLDER = Path(tempfile.mkdtemp())


def text(value):
    """Write a JSON value on one line."""
    return json.dumps(value, ensure_ascii=False)


def show(holder, node=()):
    """Print the pairs of a holder or of one of its value nodes, then their entries.

    A value node's own entries are left out of its pair; ``node=`` shows them.
    """
    for property, value in holder.get_pairs(node=node):
        if isinstance(value, dict):
            value = {key: item for key, item in value.items() if key != "provenance"}
        print(f"{property}: {text(value)}")
    for entry in holder.get_provenance(node=node):
        rest = {
            key: value
            for key, value in entry.items()
            if key not in ("property", "value", "status")
        }
        print(f"[{entry['status']}] {entry['property']}: {text(entry['value'])}")
        print(f"    {text(rest)}")


def show_manifest(manifest):
    """Print a manifest as canonical YAML, its context reduced to the prefixes."""
    prefixes = {
        key: value
        for key, value in manifest["@context"].items()
        if key not in FORMAT_CONTEXT
    }
    print(dumps({**manifest, "@context": prefixes}), end="")

The bundle holds the three timelines: the encoding `clt_6` in quarters,
27 of them for six bars of 9/8; the video `cpt_1` in seconds, of unknown
length, with a child `cpt_2` for the part the example uses; and the WAV
file `dpt_1` in samples.

In [2]:
bundle = tta.AlignmentBundle(
    id="layerit", name="LayerIt, Clair de lune mm. 1–6 (Pires)"
)
score = tta.ContinuousLogicalTimeline(
    length=27, uid="clt_6", name="Clair de lune, mm. 1–6 (MEI)"
)
video = tta.ContinuousPhysicalTimeline(
    length=None, uid="cpt_1", name="Maria João Pires live at Jardin Musical"
)
video.create_child(length=33.353016, uid="cpt_2")
wav = tta.DiscretePhysicalTimeline(
    length=1470868, uid="dpt_1", name="ClairDeLune_MariaJoaoPires_untilM6.wav"
)
for timeline in (score, video, wav):
    bundle.add_timeline(timeline)
bundle.timeline_ids

['clt_6', 'cpt_1', 'dpt_1']

## Say what the bundle and a timeline are

`add_statement()` states one pair on a timeline or a bundle, in place. The
bundle is about the work, named here by its Wikidata item; the value is a
node, a thing with an `id` and a `label` cached for display. The video's
timeline is a video: `type` takes a class, and the catalogue's verb
`is_a` states the same in Wikidata's vocabulary, as an instance of
(`wdt:P31`) the item for a video, `wd:Q98069877`. `add_verb()` takes the
verb, the system and the value, and states the pairs the catalogue's
pattern for them gives.

In [3]:
bundle.add_statement("schema:about", {"id": "wd:Q29117946", "label": "Clair de lune"})
video.add_statement("type", "schema:VideoObject")
video.add_verb("is_a", "wikidata", "wd:Q98069877")
video.statements

{'type': ['schema:VideoObject'], 'wdt:P31': [{'id': 'wd:Q98069877'}]}

## One verb, two vocabularies

The pianist is named in two systems: by a Wikidata item through
Wikidata's property "performer" (`wdt:P175`), and by a VIAF identifier
through the MARC relator "performer" (`rel:prf`), the fallback for a
system without a property of its own. Each pair holds a node of its own.

In [4]:
video.add_verb("performed_by", "wikidata", "wd:Q235650", label="Maria João Pires")
video.add_verb("performed_by", "fallback", "viaf:113931950")
[(rectangle.verb, rectangle.things) for rectangle in video.get_rectangles()]

[('is_a',
  ('http://schema.org/VideoObject',
   'http://www.wikidata.org/entity/Q98069877')),
 ('performed_by', ('http://viaf.org/viaf/113931950',)),
 ('performed_by', ('http://www.wikidata.org/entity/Q235650',))]

`get_rectangles()` gathers the pairs the catalogue reads as one verb about
one thing. So far the two performers are two things. A value node is a
subject too: `node=` addresses it by its path, the property and the
index, and `schema:sameAs` on the Wikidata node ties the two. The pairs
then form one rectangle, labelled by the node's label.

In [5]:
video.add_statement("schema:sameAs", {"id": "viaf:113931950"}, node=("wdt:P175", 0))
[
    (rectangle.verb, rectangle.label, rectangle.things)
    for rectangle in video.get_rectangles()
]

[('is_a',
  None,
  ('http://schema.org/VideoObject',
   'http://www.wikidata.org/entity/Q98069877')),
 ('performed_by',
  'Maria João Pires',
  ('http://viaf.org/viaf/113931950',
   'http://www.wikidata.org/entity/Q235650'))]

## Who said so

A pair without a provenance entry is stated by the manifest's publisher.
`provenance=` adds a `confirmed` entry naming who stated the pair, when,
how and why; the keys are those the file writes. `get_provenance()`
answers the entries, optionally of one status.

In [6]:
video.add_statement(
    "schema:sameAs",
    {"id": "https://www.youtube.com/watch?v=Ch2mrPm1JnM"},
    provenance={
        "stated_by": [EDITOR],
        "date": DAY,
        "method": "manual",
        "note": "pasted URL",
    },
)
video.get_provenance()

[{'property': 'schema:sameAs',
  'value': {'id': 'https://www.youtube.com/watch?v=Ch2mrPm1JnM'},
  'status': 'confirmed',
  'stated_by': ['https://example.org/people/editor'],
  'date': '2026-11-18',
  'method': 'manual',
  'note': 'pasted URL'}]

## Propose, confirm and reject

A proposal is a candidate awaiting a decision: it exists only as its
entry, and the subject does not state it. The WAV file was cut from the
video's sound, which is an inference. A value naming an element of the
manifest is the node of its reference, `#` followed by its ID, which
`to_file_id()` builds.

In [7]:
source = {"id": to_file_id(video.id)}
wav.propose_statement(
    "dcterms:source",
    source,
    stated_by=EDITOR,
    method="inferred",
    date=DAY,
    note="assumed ripped from the video's sound and cut",
)
show(wav)

[proposed] dcterms:source: {"id": "#cpt_1"}
    {"stated_by": ["https://example.org/people/editor"], "date": "2026-11-18", "method": "inferred", "note": "assumed ripped from the video's sound and cut"}


Confirming states the pair and turns its entry `confirmed`, with the new
agent and day and every other key kept.

In [8]:
wav.confirm_statement("dcterms:source", source, stated_by=EDITOR, date="2026-11-19")
show(wav)

dcterms:source: {"id": "#cpt_1"}
[confirmed] dcterms:source: {"id": "#cpt_1"}
    {"stated_by": ["https://example.org/people/editor"], "date": "2026-11-19", "method": "inferred", "note": "assumed ripped from the video's sound and cut"}


A rejection keeps a candidate that was examined and refused, so that no
tool proposes it again. A lookup in the GND proposes its entry "Clair de
lune" as the work the encoding is an example of; it is Debussy's song on
Verlaine's poem, not the piano piece. The rejected entry keeps how the
candidate was found.

In [9]:
score.add_statement("type", "melod:Manifestation")
score.propose_statement(
    "schema:exampleOfWork",
    {"id": "gnd:1156628156"},
    stated_by=EDITOR,
    method="reconciled",
    service="https://lobid.org/gnd/search",
    date=DAY,
)
score.reject_statement(
    "schema:exampleOfWork",
    {"id": "gnd:1156628156"},
    stated_by=EDITOR,
    date=DAY,
    note="GND 'Clair de lune' is Debussy's song L 32 on Verlaine, not the piano piece",
)
show(score)

type: "melod:Manifestation"
[rejected] schema:exampleOfWork: {"id": "gnd:1156628156"}
    {"stated_by": ["https://example.org/people/editor"], "date": "2026-11-18", "method": "reconciled", "service": "https://lobid.org/gnd/search", "note": "GND 'Clair de lune' is Debussy's song L 32 on Verlaine, not the piano piece"}


Proposals and decisions take a node path as statements do. The encoding
is an edition of the work (`wdt:P629`), and the GND entry for the piano
piece is proposed as the same work, inside that node.

In [10]:
score.add_statement("wdt:P629", {"id": "wd:Q29117946", "label": "Clair de lune"})
score.propose_statement(
    "schema:sameAs",
    {"id": "gnd:300042558"},
    node=("wdt:P629", 0),
    stated_by=EDITOR,
    method="reconciled",
    service="https://lobid.org/gnd/search",
    match="exact",
    date=DAY,
)
show(score, node=("wdt:P629", 0))

[proposed] schema:sameAs: {"id": "gnd:300042558"}
    {"stated_by": ["https://example.org/people/editor"], "date": "2026-11-18", "method": "reconciled", "service": "https://lobid.org/gnd/search", "match": "exact"}


## Describe a region

A region stands for the stretch of its timeline's manifestation it marks.
Regions are frozen: `with_statement()` returns a copy holding the pair,
and `replace_region()` stores it in the place of the region with its ID.
The region's siblings `without_statement()`, `with_proposal()`,
`with_confirmation()`, `with_rejection()`, `with_verb()` and
`with_prefix()` mirror the timeline's methods.

In [11]:
opening = score.create_region(0, 9, id="region_1", name="mm. 1–2")
score.replace_region(
    opening.with_statement("rdfs:comment", "the opening, before the melody enters")
)
opening.statements, score.get_region("region_1").statements

({}, {'rdfs:comment': ['the opening, before the melody enters']})

## Write the manifest

`to_dict()` writes each element's `statements` and `provenance` where the
element stands, and an `@context` holding the format's terms and the
prefixes the statements use, no others. Every prefix used here is the
catalogue's; a prefix it lacks is declared on the element with
`add_prefix()`, and a pair using an undeclared one is refused.

In [12]:
manifest = bundle.to_dict()
show_manifest(manifest)

"@context":
  dcterms: "http://purl.org/dc/terms/"
  gnd: "https://d-nb.info/gnd/"
  melod: "https://lod.academy/melod/vocab/ontology#"
  rdfs: "http://www.w3.org/2000/01/rdf-schema#"
  rel: "http://id.loc.gov/vocabulary/relators/"
  schema: "http://schema.org/"
  viaf: "http://viaf.org/viaf/"
  wd: "http://www.wikidata.org/entity/"
  wdt: "http://www.wikidata.org/prop/direct/"
id: "#layerit"
name: "LayerIt, Clair de lune mm. 1–6 (Pires)"
statements:
  "schema:about":
    -
      id: "wd:Q29117946"
      label: "Clair de lune"
timelines:
  -
    id: "#clt_6"
    length: [27, 1]
    name: "Clair de lune, mm. 1–6 (MEI)"
    provenance:
      -
        date: "2026-11-18"
        method: reconciled
        note: "GND 'Clair de lune' is Debussy's song L 32 on Verlaine, not the piano piece"
        property: "schema:exampleOfWork"
        service: "https://lobid.org/gnd/search"
        stated_by: ["https://example.org/people/editor"]
        status: rejected
        value:
          id: "gnd

The bundle read back writes the same manifest, and so does a manifest file.

In [13]:
bundle.to_file(FOLDER / "layerit.yaml")
(
    tta.AlignmentBundle.from_dict(manifest).to_dict() == manifest,
    tta.AlignmentBundle.from_file(FOLDER / "layerit.yaml").to_dict() == manifest,
)

(True, True)

A timeline written on its own writes a manifest of its own. The WAV
file's source names the video, which that manifest does not hold, so the
pair is left out together with its entry; the bundle's own statements
stay behind, and the `@context` holds the format's terms alone.

In [14]:
show_manifest(bundle.get_timeline("dpt_1").to_dict())

"@context": {}
timelines:
  -
    id: "#dpt_1"
    length: 1470868
    name: ClairDeLune_MariaJoaoPires_untilM6.wav
    unit: samples
timetoshare: "0.2.0"


## Import an MEI header

An MEI file's `meiHead` describes the work it encodes, the work's
expressions, the manifestations embodying them and the items exemplifying
those. `MeiHeadLoader` states that description in LRMoo's and MeLODy's
vocabulary. The header below is a short one modelled on the Carl Nielsen
Edition's header of the song *Hr. Olufs Sang*, CNW 133; the autograph's
address is invented.

In [15]:
HEADER = """<?xml version="1.0" encoding="UTF-8"?>
<mei xmlns="http://www.music-encoding.org/ns/mei" meiversion="5.1">
 <meiHead>
  <fileDesc><titleStmt><title>Hr. Olufs Sang</title></titleStmt><pubStmt/></fileDesc>
  <workList>
   <work xml:id="work_1">
    <identifier xml:id="identifier_1" label="CNW">133</identifier>
    <title xml:id="title_1" type="main" xml:lang="da">Hr. Olufs Sang</title>
    <composer><persName xml:id="persName_1" role="composer">Carl Nielsen</persName></composer>
    <lyricist><persName xml:id="persName_2"
      auth.uri="https://d-nb.info/gnd/" codedval="119018934">Holger Drachmann</persName></lyricist>
    <expressionList>
     <expression xml:id="expression_1">
      <title xml:id="title_3">Version for voice and piano</title>
      <key xml:id="key_1" pname="f" accid="s" mode="minor"/>
     </expression>
    </expressionList>
   </work>
  </workList>
  <manifestationList>
   <manifestation xml:id="source_1">
    <titleStmt><title xml:id="title_4">Score, print</title></titleStmt>
    <relationList>
     <relation xml:id="relation_2" rel="isEmbodimentOf" target="#expression_1"/>
    </relationList>
   </manifestation>
   <manifestation xml:id="source_2" sameas="https://example.org/sources/autograph">
    <titleStmt><title xml:id="title_5" type="main">Score, autograph</title></titleStmt>
    <itemList>
     <item xml:id="item_1">
      <identifier xml:id="identifier_4" label="RISM">DK-Kk CNS 342e</identifier>
     </item>
    </itemList>
   </manifestation>
  </manifestationList>
 </meiHead>
</mei>
"""
(FOLDER / "cnw0133.mei").write_text(HEADER, encoding="utf-8")
song = MeiHeadLoader.from_file(FOLDER / "cnw0133.mei").create_bundle(date="2026-11-19")
song.id, [(tl_id, song.get_timeline(tl_id).name) for tl_id in song.timeline_ids]

('cnw0133', [('cgt_1', 'Score, print'), ('cgt_2', 'Score, autograph')])

The bundle takes the file's stem as its ID. Each manifestation becomes a
graphical timeline in millimetres of unknown length, named by its title,
the axis along which pages of the print or the autograph can be laid out.
The work lands as a value node under the bundle's `schema:about`, its IRI the
file's address followed by `#` and the work's `xml:id`, and the bundle
points to the MEI file with `rdfs:seeAlso`.

In [16]:
show(song)

rdfs:seeAlso: {"id": "cnw0133.mei"}
schema:about: {"id": "cnw0133.mei#work_1", "type": ["melod:Work"], "melod:hasTitle": [{"type": ["melod:Title"], "rdfs:label": [{"@value": "Hr. Olufs Sang", "@language": "da"}], "melod:hasTitleType": [{"id": "melodTerms:MainTitle"}]}], "melod:hasIdentifier": [{"type": ["melod:Identifier"], "rdfs:label": ["CNW"], "owl:hasValue": ["133"]}], "rel:lyr": [{"id": "gnd:119018934", "label": "Holger Drachmann"}], "lrmoo:R3_is_realised_in": [{"id": "cnw0133.mei#expression_1"}]}
[confirmed] schema:about: {"id": "cnw0133.mei#work_1"}
    {"stated_by": ["https://timetoalign.github.io/"], "date": "2026-11-19", "method": "imported", "basis": ["cnw0133.mei#work_1"]}
[confirmed] rdfs:seeAlso: {"id": "cnw0133.mei"}
    {"stated_by": ["https://timetoalign.github.io/"], "date": "2026-11-19", "method": "imported", "basis": ["cnw0133.mei"]}


Every pair carries an entry: method `imported`, the agent
`timetoalign.timelines.LIBRARY_AGENT` (the library's address) unless
`stated_by=` names another, and as its basis
the MEI element it comes from. What the header encodes with an IRI is
stated: the lyricist, whom the header names by a GND number. A name
without an IRI is proposed for reconciliation: the composer.

In [17]:
WORK = ("schema:about", 0)
show(song, node=WORK)

lrmoo:R3_is_realised_in: {"id": "cnw0133.mei#expression_1"}
melod:hasIdentifier: {"type": ["melod:Identifier"], "rdfs:label": ["CNW"], "owl:hasValue": ["133"]}
melod:hasTitle: {"type": ["melod:Title"], "rdfs:label": [{"@value": "Hr. Olufs Sang", "@language": "da"}], "melod:hasTitleType": [{"id": "melodTerms:MainTitle"}]}
rel:lyr: {"id": "gnd:119018934", "label": "Holger Drachmann"}
type: "melod:Work"
[confirmed] rdf:type: {"id": "melod:Work"}
    {"stated_by": ["https://timetoalign.github.io/"], "date": "2026-11-19", "method": "imported", "basis": ["cnw0133.mei#work_1"]}
[confirmed] melod:hasTitle: {"type": ["melod:Title"], "rdfs:label": [{"@value": "Hr. Olufs Sang", "@language": "da"}], "melod:hasTitleType": [{"id": "melodTerms:MainTitle"}]}
    {"stated_by": ["https://timetoalign.github.io/"], "date": "2026-11-19", "method": "imported", "basis": ["cnw0133.mei#title_1"]}
[confirmed] melod:hasIdentifier: {"type": ["melod:Identifier"], "rdfs:label": ["CNW"], "owl:hasValue": ["133"]}
   

The expression is a value node too, under `lrmoo:R4_embodies` of the
manifestation that embodies it, the print; the autograph holds its item.

In [18]:
show(song.get_timeline("cgt_1"))

lrmoo:R4_embodies: {"id": "cnw0133.mei#expression_1", "type": ["melod:Expression"], "rdfs:label": ["Version for voice and piano"], "mus:U11_has_key": [{"type": ["melod:Key"], "melod:hasPitch": [{"id": "melodTerms:F"}], "melod:hasAccidental": [{"id": "melodTerms:Sharp"}], "melod:isInMode": [{"id": "melodTerms:Minor"}]}]}
melod:hasTitle: {"type": ["melod:Title"], "rdfs:label": ["Score, print"]}
type: "melod:Manifestation"
[confirmed] rdf:type: {"id": "melod:Manifestation"}
    {"stated_by": ["https://timetoalign.github.io/"], "date": "2026-11-19", "method": "imported", "basis": ["cnw0133.mei#source_1"]}
[confirmed] melod:hasTitle: {"type": ["melod:Title"], "rdfs:label": ["Score, print"]}
    {"stated_by": ["https://timetoalign.github.io/"], "date": "2026-11-19", "method": "imported", "basis": ["cnw0133.mei#title_4"]}
[confirmed] lrmoo:R4_embodies: {"id": "cnw0133.mei#expression_1"}
    {"stated_by": ["https://timetoalign.github.io/"], "date": "2026-11-19", "method": "imported", "basis": 

A proposal is decided as any other. Here the editor confirms the
composer's name as the header writes it; a lookup service could instead
propose the person's IRI, as the GND entry was proposed above.

In [19]:
song.confirm_statement(
    "rel:cmp", {"label": "Carl Nielsen"}, stated_by=EDITOR, date="2026-11-19", node=WORK
)
song.get_provenance("proposed", node=WORK), song.statements["schema:about"][0][
    "rel:cmp"
]

([], [{'label': 'Carl Nielsen'}])

`create_bundle(into=…)` states the description on an existing bundle, and
`subjects=` maps the `xml:id` of a manifestation or an item to one of its
timelines or regions. Importing again adds nothing: a pair already
stated, proposed or rejected is passed over, so a decision stays.
`base=` gives the MEI file's published address, from which every IRI and
basis built from an `xml:id` then begins.

## Import a MeLODy repository

A MeLODy data repository keeps one Turtle file per entity: a work, an
expression, a manifestation, an item, a person. Its statements are pairs
already, so `MelodyLoader` carries them as they are, in LRMoo, MeLODy's
own terms and schema.org. The repository below is written for this guide
in the shape of MeLODy's template.

In [20]:
PREFIXES = """\
@prefix melod: <https://lod.academy/melod/vocab/ontology#> .
@prefix melodTerms: <https://lod.academy/melod/vocab/terms#> .
@prefix lrmoo: <http://iflastandards.info/ns/lrm/lrmoo/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix schema: <https://schema.org/> .
"""
REPOSITORY = {
    "works/1.ttl": """\
<urn:uuid:works/1> a melod:Work ;
  melod:hasTitle [ a melod:Title ; rdfs:label "Clair de lune"@fr ] ;
  owl:sameAs <http://www.wikidata.org/entity/Q29117946> ;
  lrmoo:R3_is_realised_in <urn:uuid:expressions/2> .
""",
    "expressions/2.ttl": """\
<urn:uuid:expressions/2> a melod:Expression ;
  rdfs:label "Clair de lune, piano" ;
  melod:hasContribution [ a melod:Contribution ; melod:hasAgent <urn:uuid:persons/3> ;
    melod:hasRole melodTerms:Composer ] .
""",
    "persons/3.ttl": """\
<urn:uuid:persons/3> a melod:Person ; schema:givenName "Claude" ;
  schema:familyName "Debussy" ; owl:sameAs <http://www.wikidata.org/entity/Q4700> .
""",
    "manifestations/4.ttl": """\
<urn:uuid:manifestations/4> a melod:Manifestation ;
  melod:hasTitle [ a melod:Title ; rdfs:label "Suite bergamasque, first edition" ] ;
  lrmoo:R4_embodies <urn:uuid:expressions/2> .
""",
    "items/5.ttl": """\
<urn:uuid:items/5> a melod:Item ; rdfs:label "Copy 1" ; melod:hasShelfmark "TEST 1" ;
  lrmoo:R7_exemplifies <urn:uuid:manifestations/4> .
""",
}
repository = FOLDER / "melody"
for relative, turtle in REPOSITORY.items():
    path = repository / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(PREFIXES + turtle, encoding="utf-8")

suite = MelodyLoader.from_file(repository).create_bundle(
    id="clair_de_lune", date="2026-11-19"
)
[(tl_id, suite.get_timeline(tl_id).name) for tl_id in suite.timeline_ids]

[('cgt_1', 'Suite bergamasque, first edition')]

The one manifestation embodying the work's expression became a timeline.
The work is the value of the bundle's `schema:about`, with the triples of
its file. IRIs are compacted with the catalogue's prefixes, which read
MeLODy's `https://schema.org/` as `schema:`.

In [21]:
show(suite, node=("schema:about", 0))

lrmoo:R3_is_realised_in: {"id": "urn:uuid:expressions/2", "label": "Clair de lune, piano"}
melod:hasTitle: {"type": ["melod:Title"], "rdfs:label": [{"@value": "Clair de lune", "@language": "fr"}]}
owl:sameAs: {"id": "wd:Q29117946"}
type: "melod:Work"


The manifestation's timeline states its own IRI as `owl:sameAs`, the
expression it embodies as a value node and its item under
`lrmoo:R7i_is_exemplified_by`; the item names its manifestation by the
timeline's reference. A person who is no subject is a value, labelled
from the triples of its file. Each entry's basis is the file the pair was
read from.

In [22]:
show(suite.get_timeline("cgt_1"))

lrmoo:R4_embodies: {"id": "urn:uuid:expressions/2", "type": ["melod:Expression"], "melod:hasContribution": [{"type": ["melod:Contribution"], "melod:hasAgent": [{"id": "urn:uuid:persons/3", "label": "Claude Debussy", "owl:sameAs": [{"id": "wd:Q4700"}]}], "melod:hasRole": [{"id": "melodTerms:Composer"}]}], "rdfs:label": ["Clair de lune, piano"]}
lrmoo:R7i_is_exemplified_by: {"id": "urn:uuid:items/5", "type": ["melod:Item"], "lrmoo:R7_exemplifies": [{"id": "#cgt_1"}], "melod:hasShelfmark": ["TEST 1"], "rdfs:label": ["Copy 1"]}
melod:hasTitle: {"type": ["melod:Title"], "rdfs:label": ["Suite bergamasque, first edition"]}
owl:sameAs: {"id": "urn:uuid:manifestations/4"}
type: "melod:Manifestation"
[confirmed] rdf:type: {"id": "melod:Manifestation"}
    {"stated_by": ["https://timetoalign.github.io/"], "date": "2026-11-19", "method": "imported", "basis": ["manifestations/4.ttl"]}
[confirmed] owl:sameAs: {"id": "urn:uuid:manifestations/4"}
    {"stated_by": ["https://timetoalign.github.io/"], "

Both importers' bundles write manifests as any other, their `@context`
holding the prefixes of LRMoo, MeLODy and the other vocabularies used.

In [23]:
sorted(key for key in suite.to_dict()["@context"] if key not in FORMAT_CONTEXT)

['lrmoo', 'melod', 'melodTerms', 'owl', 'rdf', 'rdfs', 'schema', 'wd']

## Next

- Statements, provenance, the inline context and the catalogue as the
  format specifies them: TimeToShare's
  [description of a manifest](https://github.com/TimeToAlign/TimeToShare/blob/main/docs/description.md).
- Saving manifests to files, selecting one timeline or group, and the
  complete and minimal forms:
  [How to Save and Load Timelines, Groups and Bundles](how02_group_and_bundle_files.ipynb).
- Alignments between the timelines a bundle holds: the
  [Alignment Bundles and MatchClaims](../tutorials/tut06_alignment_bundles.ipynb)
  tutorial.